# NeuralAIRL applied workflow

This notebook fits a nonlinear state-only AIRL model, checks its behavioral surface, and re-solves the learned reward under changed dynamics.

In [1]:
from pathlib import Path
import econirl
from econirl import AIRL, NeuralAIRL

checkout = Path.cwd().resolve()
module_path = Path(econirl.__file__).resolve()
print("Installed package import:", not module_path.is_relative_to(checkout))
print("Standalone estimator:", NeuralAIRL is not AIRL)

Installed package import: True
Standalone estimator: True


In [2]:
import jax.numpy as jnp
import numpy as np
from econirl.core.bellman import SoftBellmanOperator
from econirl.core.solvers import value_iteration
from econirl.core.types import DDCProblem, Panel, Trajectory

n_states = 9
x = np.linspace(-1.0, 1.0, n_states)
state_inputs = x[:, None]
true_reward = 1.5 * np.cos(np.pi * x) - 0.35 * x
transitions = np.zeros((2, n_states, n_states))
for state in range(n_states):
    transitions[0, state, (state + 1) % n_states] = 0.9
    transitions[0, state, state] = 0.1
    transitions[1, state, (state - 1) % n_states] = 0.9
    transitions[1, state, state] = 0.1
problem = DDCProblem(n_states, 2, 0.9, 1.0)
oracle = value_iteration(
    SoftBellmanOperator(problem, jnp.asarray(transitions)),
    jnp.repeat(jnp.asarray(true_reward)[:, None], 2, axis=1),
    tol=1e-10,
    max_iter=5_000,
)
rng = np.random.default_rng(26_001)
trajectories = []
for individual in range(80):
    current = int(rng.integers(n_states))
    states, actions, next_states = [], [], []
    for _ in range(20):
        chosen = int(rng.choice(2, p=np.asarray(oracle.policy[current])))
        following = int(rng.choice(n_states, p=transitions[chosen, current]))
        states.append(current); actions.append(chosen); next_states.append(following)
        current = following
    trajectories.append(Trajectory(jnp.asarray(states), jnp.asarray(actions), jnp.asarray(next_states), individual_id=individual))
panel = Panel(trajectories=trajectories)
print("Panel:", panel.num_individuals, "trajectories,", panel.num_observations, "transitions")

Panel: 80 trajectories, 1600 transitions


In [3]:
model = NeuralAIRL(
    n_states=n_states,
    n_actions=2,
    discount=0.9,
    feature_matrix=state_inputs,
    reward_hidden_dim=32,
    reward_num_layers=2,
    shaping_hidden_dim=32,
    policy_hidden_dim=32,
    policy_steps=15,
    discriminator_steps=3,
    max_rounds=160,
    min_rounds=70,
    policy_step_size=0.1,
    compute_se=False,
    seed=26_002,
).fit(panel, transitions=transitions)
print("Converged:", model.converged_)
print("Rounds:", model.n_iter_)

Converged: True
Rounds: 90


In [4]:
policy_tv = 0.5 * np.abs(model.policy_ - oracle.policy).sum(axis=1).mean()
changed = transitions.copy()
for state in range(n_states):
    changed[0, state] = 0.0
    changed[0, state, (state + 1) % n_states] = 0.7
    changed[0, state, state] = 0.3
counterfactual = model.counterfactual(transitions=changed)
policy_shift = 0.5 * np.abs(counterfactual.policy_change).sum(axis=1).mean()
print(f"Policy TV to oracle: {policy_tv:.4f}")
print(f"Changed-dynamics policy shift: {policy_shift:.4f}")

Policy TV to oracle: 0.0156
Changed-dynamics policy shift: 0.0460


## Interpretation

The estimator recovered a nonlinear behavioral policy from one-dimensional state inputs. The changed-dynamics result re-solves the learned reward. It does not treat network weights as structural parameters.